# Swiss Roll로 배우는 PCA / LDA / MDS 비교

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gshong-ai/ADA26/blob/claude/mnist-dimensionality-reduction-H2Meg/02_pca_lda_mds_comparison/pca_lda_mds_comparison.ipynb)

## 개요

**Swiss Roll**은 2D 평면이 나선형으로 말린 3D 구조입니다.  
동일한 데이터에 PCA / LDA / 고전 MDS / Isomap을 적용하여  
각 기법의 **성공·실패 원인**을 해부합니다.

| 기법 | 핵심 가정 | 지도 | Swiss Roll |
|------|----------|:----:|:----------:|
| **PCA** | 분산이 큰 방향 = 의미 있는 방향 | ❌ | ✗ 접힌 층이 겹침 |
| **LDA** | 레이블 있는 클래스를 선형으로 분리 | ✅ | ✗ 층 구분만, 언폴딩 불가 |
| **고전 MDS** | 유클리드 쌍별 거리 보존 | ❌ | ✗ PCA와 유사 결과 |
| **Isomap** | **측지 거리** 보존 (k-NN 최단 경로) | ❌ | ✓ 매니폴드 언폴딩 |

### 핵심 질문
> Swiss Roll에서 PCA와 고전 MDS는 왜 실패하고, Isomap은 왜 성공할까?  
> → **유클리드 거리 ≠ 측지 거리** (접힌 매니폴드의 함정)

## 0. 패키지 설치

In [ ]:
!pip install -q plotly scikit-learn scipy

## 1. 라이브러리 임포트

In [ ]:
# ─── 라이브러리 임포트 ───
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import MDS, Isomap
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

from scipy.sparse.csgraph import shortest_path

pio.renderers.default = 'colab'
print('라이브러리 로드 완료')

## 2. 하이퍼파라미터 & 설정

In [ ]:
# ─── 하이퍼파라미터 & 전역 설정 ───
RANDOM_SEED        = 42
N_SAMPLES          = 1000      # Swiss Roll 샘플 수 (MDS 연산 시간 고려)
SWISS_NOISE        = 0.1       # 노이즈 (작게 → 매니폴드 구조 선명)

# ─── LDA 설정 ───
N_LDA_CLASSES      = 6         # t를 몇 등분할지 (LDA 클래스 레이블 생성용)

# ─── Isomap 설정 ───
ISOMAP_N_NEIGHBORS = 10        # k-NN 이웃 수 (너무 작으면 그래프 단절, 너무 크면 지름길)

# ─── Shepard 다이어그램용 서브샘플 ───
DIST_SAMPLE        = 400       # 거리 행렬 계산 샘플 수 (전체 n² 계산 회피)

# ─── 시각화 ───
COLORSCALE         = 'Rainbow'
OUTPUT_HTML        = 'pca_lda_mds_comparison.html'

np.random.seed(RANDOM_SEED)
print('설정 완료')

## 3. 데이터 로드 & 전처리

`make_swiss_roll`이 반환하는 `t`는 **나선 위치 (매니폴드 파라미터)** 입니다.  
색상이 `t`에 따라 **무지개처럼 연속적**이면 매니폴드 구조가 잘 보존된 것입니다.

In [ ]:
# ─── Swiss Roll 생성 ───
X_raw, t = make_swiss_roll(n_samples=N_SAMPLES, noise=SWISS_NOISE,
                            random_state=RANDOM_SEED)

# 표준화 (PCA/LDA/MDS 학습 안정성)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw).astype('float32')

# ─── LDA 클래스 레이블: t를 N_LDA_CLASSES 등분 ───
# percentile 경계로 균등 분할
bins = np.percentile(t, np.linspace(0, 100, N_LDA_CLASSES + 1)[1:-1])
y_lda = np.digitize(t, bins=bins)      # 0 ~ N_LDA_CLASSES-1

print(f'Swiss Roll shape   : {X.shape}')
print(f't 범위             : {t.min():.2f} ~ {t.max():.2f}')
print(f'LDA 클래스 수       : {N_LDA_CLASSES}')
print(f'클래스별 샘플 수    : {np.bincount(y_lda).tolist()}')

# ─── 3D 원본 시각화 ───
fig_3d = go.Figure(go.Scatter3d(
    x=X_raw[:, 0], y=X_raw[:, 1], z=X_raw[:, 2],
    mode='markers',
    marker=dict(size=3, color=t, colorscale=COLORSCALE, opacity=0.85,
                colorbar=dict(title='t (매니폴드 위치)')),
))
fig_3d.update_layout(
    title='원본 Swiss Roll (3D) — 색상이 연속적일수록 구조 보존이 잘 된 것',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    width=700, height=530,
)
fig_3d.show()

## 4. PCA — 분산이 큰 방향으로 선형 투영

### 이론
```
공분산 행렬 C = (1/n) Xᵀ X
고유값 분해: C = V Λ Vᵀ
투영:        z = V[:,0:2]ᵀ · x
```

PC1, PC2는 **데이터의 분산을 가장 많이 설명하는 방향**입니다.  
Swiss Roll은 나선형이므로 분산이 큰 방향이 나선 축 방향 → 인접한 나선 층이 겹칩니다.

### Swiss Roll에서의 실패 원인
선형 변환은 **나선의 곡률**을 인식하지 못합니다.  
같은 나선 층이라도 위치에 따라 3D 좌표가 다르지만, PCA는 이를 한 방향으로 "눌러" 버립니다.

In [ ]:
# ─── PCA 적용 ───
pca = PCA(n_components=2, random_state=RANDOM_SEED)
coords_pca = pca.fit_transform(X)

expl = pca.explained_variance_ratio_
print(f'PC1 설명 분산: {expl[0]:.3f} ({expl[0]*100:.1f}%)')
print(f'PC2 설명 분산: {expl[1]:.3f} ({expl[1]*100:.1f}%)')
print(f'합계          : {expl.sum():.3f} ({expl.sum()*100:.1f}%)')

# ─── PC 벡터 (3D 원본 공간에서의 주성분 방향) ───
print(f'\nPC1 벡터 (3D): {pca.components_[0].round(3)}')
print(f'PC2 벡터 (3D): {pca.components_[1].round(3)}')

# ─── 2D 투영 시각화 ───
fig_pca = go.Figure(go.Scatter(
    x=coords_pca[:, 0], y=coords_pca[:, 1],
    mode='markers',
    marker=dict(size=4, color=t, colorscale=COLORSCALE, opacity=0.8,
                colorbar=dict(title='t')),
))
fig_pca.update_layout(
    title='PCA 2D 투영 — 색상이 섞이면 인접 층이 겹쳐 구조 파괴',
    xaxis_title='PC1', yaxis_title='PC2',
    width=600, height=450,
)
fig_pca.show()
print('\n⚠️  색상이 섞여 있으면 PCA가 Swiss Roll을 펴지 못한 것입니다.')

## 5. LDA — 클래스 간 분산 / 클래스 내 분산 최대화

### 이론
```
SB = Σ nk (μk - μ)(μk - μ)ᵀ   (클래스 간 산포 행렬)
SW = Σ Σ (xᵢ - μk)(xᵢ - μk)ᵀ (클래스 내 산포 행렬)
목적: max  wᵀ SB w / wᵀ SW w
→ 일반화 고유값 문제: SW⁻¹ SB w = λ w
```

### LDA가 Swiss Roll에서 실패하는 이유

LDA는 **선형 판별 경계**를 학습합니다.  
Swiss Roll을 t 값으로 등분하면 각 클래스는 **나선의 호(arc)** 형태입니다.  
이웃한 나선 층은 3D 좌표가 겹치므로 선형 경계로는 층을 "펼칠" 수 없습니다.

> 💡 설령 정확한 t 레이블을 줘도 LDA는 언폴딩에 실패합니다.  
> 실패 원인이 레이블 부재가 아닌 **선형성 가정** 자체임을 보여줍니다.

In [ ]:
# ─── LDA 클래스 레이블 시각화 ───
LDA_PALETTE = [
    '#e63946', '#f4a261', '#2a9d8f', '#457b9d', '#9b59b6', '#1abc9c'
]

fig_label = go.Figure()
for cls in range(N_LDA_CLASSES):
    mask = y_lda == cls
    fig_label.add_trace(go.Scatter3d(
        x=X_raw[mask, 0], y=X_raw[mask, 1], z=X_raw[mask, 2],
        mode='markers',
        marker=dict(size=3, color=LDA_PALETTE[cls], opacity=0.8),
        name=f'클래스 {cls}',
    ))

fig_label.update_layout(
    title=f'LDA 클래스 레이블 (t를 {N_LDA_CLASSES}등분) — 각 색상이 나선의 한 "호(arc)"에 해당',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    width=700, height=530,
)
fig_label.show()
print('각 클래스는 나선을 따라 연속된 구간이지만, 다른 나선 층과 3D 공간에서 겹칩니다.')

In [ ]:
# ─── LDA 적용 ───
# n_components <= min(n_features, n_classes-1) = min(3, 5) = 3 → 2 사용
lda = LDA(n_components=2)
coords_lda = lda.fit_transform(X, y_lda)

print('=== LDA 결과 ===')
print(f'투영 차원: {coords_lda.shape}')
print(f'클래스 간 분리도 (explained variance ratio): {lda.explained_variance_ratio_.round(4)}')

# 클래스 평균 간 거리 확인
class_means = lda.means_
print(f'\n클래스 평균 쌍별 거리 (2D 투영 공간):')
for i in range(N_LDA_CLASSES):
    for j in range(i+1, N_LDA_CLASSES):
        d = np.linalg.norm(class_means[i] - class_means[j])
        print(f'  클래스 {i} ↔ 클래스 {j}: {d:.4f}')

# ─── 2D 투영 시각화 (두 가지 색상 코딩) ───
fig_lda = make_subplots(
    rows=1, cols=2,
    subplot_titles=['색상 = t (매니폴드 위치)', f'색상 = LDA 클래스 ({N_LDA_CLASSES}등분)'],
)

# 왼쪽: t로 색상 (언폴딩 여부 확인)
fig_lda.add_trace(go.Scatter(
    x=coords_lda[:, 0], y=coords_lda[:, 1],
    mode='markers',
    marker=dict(size=4, color=t, colorscale=COLORSCALE, opacity=0.8),
    showlegend=False,
), row=1, col=1)

# 오른쪽: 클래스로 색상 (클래스 분리 확인)
for cls in range(N_LDA_CLASSES):
    mask = y_lda == cls
    fig_lda.add_trace(go.Scatter(
        x=coords_lda[mask, 0], y=coords_lda[mask, 1],
        mode='markers',
        marker=dict(size=4, color=LDA_PALETTE[cls], opacity=0.8),
        name=f'클래스 {cls}',
    ), row=1, col=2)

fig_lda.update_layout(
    title='LDA 2D 투영 — 클래스 분리는 되지만 나선 구조는 펼쳐지지 않음',
    width=1000, height=430,
)
fig_lda.show()
print('\n왼쪽: 색상이 섞여있음 → 나선 층이 겹쳐 언폴딩 실패')
print('오른쪽: 클래스별로 분리 → LDA 본연의 목표(클래스 분리)는 달성')

## 6. 고전 MDS — 유클리드 쌍별 거리 보존

### 이론

```
입력: 거리 행렬 D  (dᵢⱼ = ‖xᵢ - xⱼ‖₂)
목적: min Σᵢⱼ (dᵢⱼ - d̂ᵢⱼ)²   (Stress)

고전 MDS (metric): 이중 중심화 (Double Centering) 후 고유값 분해
  B = -½ H D² H,  H = I - (1/n)11ᵀ
  투영: Z = [√λ₁v₁ | √λ₂v₂]
```

### Swiss Roll에서의 실패 원인

고전 MDS는 **유클리드 거리**를 사용합니다.  
Swiss Roll에서는 서로 다른 나선 층의 두 점이 **3D 거리는 작지만 매니폴드 거리(측지)는 큽니다**.  
MDS는 이 두 점을 2D에서도 가깝게 배치 → 나선이 접힌 채로 투영됩니다.

> 결과: 고전 MDS는 PCA와 매우 유사한 투영을 만듭니다 (둘 다 유클리드 기반이기 때문).

In [ ]:
# ─── 고전 MDS 적용 ───
# metric=True → Stress 최소화 (SMACOF 알고리즘)
# ⚠️ n=1000 기준으로 1~2분 소요될 수 있습니다
print('고전 MDS 계산 중... (잠시 기다려 주세요)')
cmds = MDS(n_components=2, metric=True, random_state=RANDOM_SEED,
           n_init=4, max_iter=300, normalized_stress='auto')
coords_cmds = cmds.fit_transform(X)

print(f'Stress: {cmds.stress_:.4f}  (낮을수록 거리 보존 잘 됨)')

# ─── PCA와 고전 MDS 결과 비교 ───
fig_cmds = make_subplots(
    rows=1, cols=2,
    subplot_titles=['PCA', '고전 MDS (Euclidean)'],
)

for col, (name, coords) in enumerate(
    [('PCA', coords_pca), ('고전 MDS', coords_cmds)], 1
):
    fig_cmds.add_trace(go.Scatter(
        x=coords[:, 0], y=coords[:, 1],
        mode='markers',
        marker=dict(size=4, color=t, colorscale=COLORSCALE, opacity=0.8,
                    showscale=(col == 2),
                    colorbar=dict(title='t') if col == 2 else None),
        showlegend=False,
    ), row=1, col=col)

fig_cmds.update_layout(
    title='PCA vs 고전 MDS — 유클리드 기반 두 기법의 결과는 매우 유사',
    width=1000, height=430,
)
fig_cmds.show()
print('\n고전 MDS와 PCA 결과가 유사한 이유: 둘 다 유클리드 거리 기반이기 때문입니다.')
print('유클리드 거리가 매니폴드 구조를 반영하지 못하는 것이 실패의 근본 원인입니다.')

## 7. Isomap — 측지 거리를 사용하는 MDS

### 이론

Isomap = 측지 거리(Geodesic distance)로 고전 MDS를 수행하는 알고리즘입니다.

```
[1단계] k-NN 그래프 구축
        각 점에서 k개의 가장 가까운 이웃을 연결
        (엣지 가중치 = 유클리드 거리)

[2단계] 최단 경로 = 측지 거리 근사
        모든 점 쌍에 대해 Dijkstra / Floyd-Warshall로 최단 경로 계산
        → D_geodesic 행렬

[3단계] 고전 MDS로 2D 투영
        D_geodesic을 거리 행렬로 사용 → 측지 구조 보존
```

### k-NN 이웃 수의 영향
- **너무 작으면** → 그래프가 단절 → 측지 거리가 무한대인 쌍 발생
- **너무 크면** → 다른 나선 층에 "지름길" 생김 → 측지 거리 과소 추정

In [ ]:
# ─── Isomap 적용 ───
print(f'Isomap 계산 중... (n_neighbors={ISOMAP_N_NEIGHBORS})')
isomap = Isomap(n_components=2, n_neighbors=ISOMAP_N_NEIGHBORS)
coords_iso = isomap.fit_transform(X)

print(f'Isomap 재구성 오차: {isomap.reconstruction_error():.4f}')

fig_iso = go.Figure(go.Scatter(
    x=coords_iso[:, 0], y=coords_iso[:, 1],
    mode='markers',
    marker=dict(size=4, color=t, colorscale=COLORSCALE, opacity=0.8,
                colorbar=dict(title='t (매니폴드 위치)')),
))
fig_iso.update_layout(
    title='Isomap 2D 투영 — 색상이 무지개처럼 연속적이면 언폴딩 성공 ✓',
    xaxis_title='Isomap 1', yaxis_title='Isomap 2',
    width=600, height=470,
)
fig_iso.show()

# ─── 이웃 수별 언폴딩 품질 비교 ───
print('\n=== k-NN 이웃 수별 Isomap 재구성 오차 ===')
for k in [5, 10, 15, 20]:
    iso_k = Isomap(n_components=2, n_neighbors=k)
    iso_k.fit_transform(X)
    print(f'  n_neighbors={k:2d}: {iso_k.reconstruction_error():.4f}')

## 8. 거리 왜곡 분석 — Shepard 다이어그램

PCA·고전 MDS 실패의 **근본 원인**을 정량적으로 확인합니다.

```
Shepard 다이어그램:
  x축 = 3D 유클리드 거리 (기존 기법이 사용하는 것)
  y축 = 측지 거리 (k-NN 최단 경로, Isomap이 사용하는 것)

대각선 위: 측지 거리 >> 유클리드 거리
           → 두 점이 3D에서는 가깝지만 매니폴드 위에서는 멀다
           → "접힌 부분"의 서로 다른 나선 층에 있는 점 쌍
           → PCA/MDS가 이 점들을 잘못 배치함
```

In [ ]:
# ─── 서브샘플로 거리 행렬 계산 (메모리 절약) ───
rng = np.random.default_rng(RANDOM_SEED)
idx_sub = rng.choice(N_SAMPLES, DIST_SAMPLE, replace=False)
X_sub   = X[idx_sub]
t_sub   = t[idx_sub]

# 유클리드 거리 행렬 (3D)
d_euclid = np.sqrt(
    np.sum((X_sub[:, None, :] - X_sub[None, :, :]) ** 2, axis=-1)
)  # (DIST_SAMPLE, DIST_SAMPLE)

# k-NN 그래프 → 측지 거리 (최단 경로)
nbrs_sub = NearestNeighbors(n_neighbors=ISOMAP_N_NEIGHBORS)
nbrs_sub.fit(X_sub)
knn_graph_sub = nbrs_sub.kneighbors_graph(X_sub, mode='distance')
d_geodesic    = shortest_path(knn_graph_sub, directed=False)

# ─── 상삼각 추출 (중복 제거) ───
triu = np.triu_indices(DIST_SAMPLE, k=1)
eu_vals  = d_euclid[triu]
geo_vals = d_geodesic[triu]

# inf 값 제거 (그래프 단절로 인한)
valid = np.isfinite(geo_vals) & np.isfinite(eu_vals)
eu_vals  = eu_vals[valid]
geo_vals = geo_vals[valid]

# 비율: 측지 / 유클리드 (>> 1 이면 "접힌 부분" 쌍)
ratio = geo_vals / (eu_vals + 1e-8)

print(f'전체 쌍 수: {len(eu_vals):,}')
print(f'비율 중앙값 (측지/유클리드): {np.median(ratio):.2f}')
print(f'비율 90th percentile: {np.percentile(ratio, 90):.2f}')
print(f'비율 > 3인 쌍 비율: {(ratio > 3).mean():.1%}  ← 이 쌍들이 PCA/MDS를 오도')

# ─── Shepard 다이어그램 ───
vmax_ratio = np.percentile(ratio, 95)

# 색상이 진할수록 PCA/MDS에게 "위험한" 쌍
fig_shep = go.Figure()
fig_shep.add_trace(go.Scatter(
    x=eu_vals, y=geo_vals,
    mode='markers',
    marker=dict(
        size=2, opacity=0.35,
        color=ratio,
        colorscale='RdYlGn_r',
        cmin=1.0, cmax=vmax_ratio,
        colorbar=dict(title='측지/유클리드<br>거리 비율'),
    ),
    name='점 쌍',
))

# 이상적 대각선 (유클리드 = 측지)
max_eu = np.percentile(eu_vals, 98)
fig_shep.add_trace(go.Scatter(
    x=[0, max_eu], y=[0, max_eu],
    mode='lines',
    line=dict(color='black', dash='dash', width=1.5),
    name='유클리드 = 측지 (이상)',
))

fig_shep.update_layout(
    title='Shepard 다이어그램 — 대각선 위 점(빨강): 유클리드<<측지 "접힌 쌍"<br>'
          '<sup>이 쌍들을 PCA/고전 MDS가 2D에서 가깝게 놓기 때문에 구조가 붕괴됨</sup>',
    xaxis_title='3D 유클리드 거리',
    yaxis_title='측지 거리 (k-NN 최단 경로)',
    width=700, height=520,
    legend=dict(x=0.02, y=0.98),
)
fig_shep.show()

## 9. 종합 비교 시각화

4가지 기법의 2D 투영을 나란히 비교합니다.  
**색상(t)이 무지개처럼 연속적**일수록 매니폴드 구조가 잘 보존된 것입니다.

In [ ]:
# ─── 2×2 비교 그리드 ───
plot_items = [
    ('PCA\n(선형, 비지도)',         coords_pca,  'PC1',      'PC2'),
    ('LDA\n(선형, 지도)',            coords_lda,  'LDA1',     'LDA2'),
    ('고전 MDS\n(유클리드 거리)',    coords_cmds, 'MDS1',     'MDS2'),
    ('Isomap\n(측지 거리)',          coords_iso,  'Isomap1',  'Isomap2'),
]

fig_all = make_subplots(
    rows=2, cols=2,
    subplot_titles=[label.replace('\n', ' ') for label, *_ in plot_items],
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
)

positions = [(1,1), (1,2), (2,1), (2,2)]

for (label, coords, ax1, ax2), (row, col) in zip(plot_items, positions):
    show_bar = (row == 1 and col == 2)
    fig_all.add_trace(
        go.Scatter(
            x=coords[:, 0], y=coords[:, 1],
            mode='markers',
            marker=dict(
                size=4, color=t, colorscale=COLORSCALE, opacity=0.8,
                showscale=show_bar,
                colorbar=dict(title='t', x=1.02) if show_bar else None,
            ),
            showlegend=False,
        ),
        row=row, col=col,
    )

fig_all.update_layout(
    title=dict(
        text='Swiss Roll 2D 투영 비교 — 색상(t)이 무지개처럼 연속적일수록 언폴딩 성공<br>'
             '<sup>PCA·고전 MDS: 유클리드 거리 기반 → 실패 | '
             'LDA: 선형 경계 → 실패 | Isomap: 측지 거리 → 성공</sup>',
        x=0.5, font=dict(size=15),
    ),
    width=1000, height=820,
    paper_bgcolor='white',
)
fig_all.show()

fig_all.write_html(OUTPUT_HTML, include_plotlyjs='cdn')
print(f'저장 완료: {OUTPUT_HTML}')

## 10. 정량 평가

### 평가 지표 1: 이웃 t값 분산 (연속성 점수)
2D 투영에서 k-최근접 이웃의 매니폴드 파라미터 t 분산 평균입니다.
- **낮을수록** → 2D에서 가까운 점들이 매니폴드에서도 가까움 → 잘 펼쳐진 것

### 평가 지표 2: 실루엣 스코어
t 등분 클래스 기준으로 2D 투영의 클러스터 분리도를 측정합니다.

In [ ]:
# ─── 연속성 점수 계산 ───
def continuity_score(coords_2d, t_vals, k=10):
    """k-이웃 t값 분산 평균 (낮을수록 매니폴드 구조 잘 보존)"""
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords_2d)
    _, indices = nbrs.kneighbors(coords_2d)
    return np.mean([np.var(t_vals[indices[i, 1:]]) for i in range(len(coords_2d))])

# ─── 전체 평가 ───
eval_items = [
    ('PCA',       coords_pca,  '#95a5a6'),
    ('LDA',       coords_lda,  '#3498DB'),
    ('고전 MDS',  coords_cmds, '#E74C3C'),
    ('Isomap',    coords_iso,  '#2ECC71'),
]

print('=== 정량 평가 결과 ===')
print(f'{"기법":10s}  {"이웃 t 분산":>14s}  {"실루엣":>10s}')
print('-' * 42)

cont_scores = []
sil_scores  = []
for name, coords, _ in eval_items:
    cont = continuity_score(coords, t, k=10)
    sil  = silhouette_score(coords, y_lda)
    cont_scores.append(cont)
    sil_scores.append(sil)
    print(f'{name:10s}  {cont:14.4f}  {sil:10.4f}')

print()
best_cont = eval_items[np.argmin(cont_scores)][0]
best_sil  = eval_items[np.argmax(sil_scores)][0]
print(f'이웃 t 분산 최소 (최고): {best_cont}')
print(f'실루엣 스코어 최대 (최고): {best_sil}')

# ─── 비교 막대 차트 ───
fig_eval = make_subplots(
    rows=1, cols=2,
    subplot_titles=['이웃 t값 분산 (↓ 낮을수록 좋음)', '실루엣 스코어 (↑ 높을수록 좋음)'],
)

names  = [name for name, _, _ in eval_items]
colors = [color for _, _, color in eval_items]

fig_eval.add_trace(go.Bar(
    x=names, y=cont_scores,
    marker_color=colors, opacity=0.85,
    text=[f'{s:.4f}' for s in cont_scores], textposition='auto',
    showlegend=False,
), row=1, col=1)

fig_eval.add_trace(go.Bar(
    x=names, y=sil_scores,
    marker_color=colors, opacity=0.85,
    text=[f'{s:.4f}' for s in sil_scores], textposition='auto',
    showlegend=False,
), row=1, col=2)

fig_eval.update_layout(
    title='Swiss Roll 2D 투영 정량 평가',
    width=900, height=420,
)
fig_eval.show()

## 11. 결론

### 왜 유클리드 거리가 Swiss Roll에서 실패하는가?

```
   나선 층 A ──────────────── 나선 층 B
   (t ≈ 7)                   (t ≈ 14)

        ↑ 유클리드 거리: 매우 짧음 (3D 좌표 가까움)
        ↑ 측지   거리: 매우 김  (나선 따라 돌아가야)

   PCA/MDS → 이 두 점을 2D에서 가깝게 배치 → 층이 겹쳐 보임
   Isomap  → 이 두 점을 2D에서 멀게 배치  → 나선이 펼쳐짐
```

### 각 기법 요약

| 기법 | Swiss Roll 결과 | 실패/성공 원인 |
|------|:--------------:|---------------|
| **PCA** | ✗ 실패 | 선형 투영 → 나선 곡률 인식 불가 |
| **LDA** | ✗ 실패 | 선형 판별 경계 → 층은 구분하되 펼치지 못함 |
| **고전 MDS** | ✗ 실패 | 유클리드 거리 ≠ 측지 거리 → 접힌 층 쌍 오배치 |
| **Isomap** | ✓ 성공 | k-NN 최단 경로로 측지 거리 근사 → 매니폴드 구조 보존 |

### 핵심 교훈

> **"어떤 거리를 보존하느냐"가 차원 축소의 성패를 결정합니다.**  
>
> 데이터가 매니폴드 위에 있다면 유클리드 거리가 아닌 **측지 거리**를 사용해야 합니다.  
> Isomap은 이를 k-NN 그래프의 최단 경로로 근사하여 해결합니다.

### 더 나아가기
- **LLE (Locally Linear Embedding)**: 이웃 관계로 국소 선형 근사 → 매니폴드 학습
- **t-SNE / UMAP**: 확률적 이웃 구조 보존 → 고차원 시각화에 특화
- **오토인코더**: 신경망으로 비선형 매니폴드 학습 → `04_autoencoder_manifold` 참고